In [ ]:
"""
================================================================================
 GENERADOR DE DATOS SINTÉTICOS - "BOTICA SALUD TOTAL" (empresa ficticia)
================================================================================
Genera un modelo dimensional (estrella) completo para una cadena de boticas:

    Dim_Cliente, Dim_Producto, Dim_Tienda, Dim_Tiempo, Dim_Promocion,
    Fact_Ventas

El script crea DOS versiones de cada tabla:

    data/raw/        -> versión "tal cual llegaría de un sistema operativo",
                         con problemas de calidad introducidos a propósito
                         (fechas mixtas, nulos, duplicados, texto sucio,
                         typos, outliers, claves huérfanas).
    data/processed/  -> versión limpia de las tablas Dim_* (curada / ground
                         truth), útil como referencia para validar el proceso
                         de limpieza. Dim_Cliente incluye además columnas
                         derivadas de abandono (churn) para clasificación.

Todo es 100% REPRODUCIBLE: se fija la semilla 42 en numpy, random y Faker.

Autor: Script de Data Engineering
================================================================================
"""

import os
import unicodedata
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

# ==============================================================================
# 0. CONFIGURACIÓN GENERAL Y SEMILLAS (REPRODUCIBILIDAD)
# ==============================================================================
SEED = 42
np.random.seed(SEED)

fake = Faker("es_ES")
Faker.seed(SEED)
rng = np.random.default_rng(SEED)          # generador moderno de numpy
py_random = np.random.RandomState(SEED)    # generador "clásico" (np.random.*)

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Volúmenes ----------------------------------------------------------------
N_CLIENTES = 5000
N_PRODUCTOS = 500
N_TIENDAS = 15
N_DIAS = 730                # 2 años
N_PROMOCIONES = 40
N_VENTAS = 60000            # líneas de venta objetivo (antes de duplicados)

# --- Ventana temporal del negocio ---------------------------------------------
FECHA_INICIO = datetime(2024, 1, 1)
FECHA_FIN = FECHA_INICIO + timedelta(days=N_DIAS - 1)   # 2025-12-31

print("=" * 80)
print(" GENERANDO DATOS SINTÉTICOS - BOTICA SALUD TOTAL")
print(f" Periodo de negocio: {FECHA_INICIO.date()} -> {FECHA_FIN.date()}  ({N_DIAS} días)")
print("=" * 80)


# ==============================================================================
# Funciones auxiliares genéricas
# ==============================================================================
def strip_accents(texto: str) -> str:
    """Quita tildes/acentos de un string (para simular errores de tipeo)."""
    if not isinstance(texto, str):
        return texto
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def fecha_a_texto_mixto(fecha: pd.Timestamp) -> str:
    """
    Convierte una fecha a UNO de tres formatos posibles, elegido al azar:
      - dd/mm/aaaa
      - aaaa-mm-dd
      - texto libre en español ("15 de marzo de 2024")
    Esto simula la inconsistencia típica de fuentes operativas distintas.
    """
    meses_es = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
                "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
    opcion = py_random.choice(["dd/mm/aaaa", "aaaa-mm-dd", "texto"], p=[0.45, 0.45, 0.10])
    if opcion == "dd/mm/aaaa":
        return fecha.strftime("%d/%m/%Y")
    elif opcion == "aaaa-mm-dd":
        return fecha.strftime("%Y-%m-%d")
    else:
        return f"{fecha.day} de {meses_es[fecha.month - 1]} de {fecha.year}"

    
def guardar_csv(df: pd.DataFrame, ruta: str):
    df.to_csv(ruta, sep=";", encoding="utf-8", index=False)
    print(f"   -> Guardado: {ruta}  ({len(df):,} filas)")


# ==============================================================================
# 1. DIM_TIEMPO  (~730 días)
# ==============================================================================
print("\n[1/6] Generando Dim_Tiempo ...")

fechas = pd.date_range(start=FECHA_INICIO, periods=N_DIAS, freq="D")

dias_semana_es = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

# Feriados fijos "tipo Perú" (empresa ficticia) - mismas fechas cada año dentro del rango
feriados_mm_dd = {(1, 1), (5, 1), (6, 7), (7, 28), (7, 29), (8, 30), (10, 8), (11, 1), (12, 8), (12, 25)}

dim_tiempo = pd.DataFrame({
    "fecha": fechas,
    "dia": fechas.day,
    "mes": fechas.month,
    "trimestre": fechas.quarter,
    "anio": fechas.year,
    "dia_semana": [dias_semana_es[d.weekday()] for d in fechas],
    "es_feriado": [(d.month, d.day) in feriados_mm_dd for d in fechas],
})

# --- Pesos de estacionalidad (se reutilizan más abajo para Fact_Ventas) -------
# Julio y diciembre: temporada alta. Febrero y abril: temporada baja.
peso_mes = {1: 1.0, 2: 0.65, 3: 0.85, 4: 0.65, 5: 0.95, 6: 1.05,
            7: 1.8, 8: 1.0, 9: 0.9, 10: 0.95, 11: 1.1, 12: 1.85}
peso_estacional_dia = np.array([peso_mes[m] for m in dim_tiempo["mes"]], dtype=float)
# pequeño extra en fin de semana (más visitas a tienda física)
peso_estacional_dia *= np.where(fechas.weekday >= 5, 1.15, 1.0)

dim_tiempo_clean = dim_tiempo.copy()  # Dim_Tiempo no tiene problemas de calidad definidos


# ==============================================================================
# 2. DIM_TIENDA (~15 tiendas)
# ==============================================================================
print("[2/6] Generando Dim_Tienda ...")

regiones_ciudades = [
    ("Lima Metropolitana", "Lima"), ("Lima Metropolitana", "Lima"),
    ("Lima Metropolitana", "Lima"), ("Lima Metropolitana", "Lima"),
    ("Lima Metropolitana", "Lima"), ("Norte", "Trujillo"),
    ("Norte", "Chiclayo"), ("Norte", "Piura"), ("Sur", "Arequipa"),
    ("Sur", "Tacna"), ("Sur", "Ica"), ("Centro", "Huancayo"),
    ("Oriente", "Iquitos"), ("Oriente", "Pucallpa"), ("Online", "Lima"),
]

nombres_tienda = [f"Botica Salud Total - {ciudad} {i+1}" if canal != "Online" else
                   f"Botica Salud Total - Tienda Online"
                   for i, (canal, ciudad) in enumerate(regiones_ciudades)]

dim_tienda = pd.DataFrame({
    "id_tienda": range(1, N_TIENDAS + 1),
    "nombre": nombres_tienda,
    "canal": ["online" if region == "Online" else "fisico" for region, ciudad in regiones_ciudades],
    "region": [r for r, c in regiones_ciudades],
    "ciudad": [c for r, c in regiones_ciudades],
})
dim_tienda_clean = dim_tienda.copy()  # Sin problemas de calidad definidos para esta tabla


# ==============================================================================
# 3. DIM_PRODUCTO (~500 productos) - con concentración tipo Pareto en ventas
# ==============================================================================
print("[3/6] Generando Dim_Producto ...")

catalogo_categorias = {
    "Medicamentos": {
        "subcategorias": ["Analgésicos", "Antigripales", "Antibióticos",
                           "Gastrointestinales", "Antialérgicos", "Antiinflamatorios", "Cardiovascular"],
        "nombres": ["Paracetamol 500mg", "Ibuprofeno 400mg", "Amoxicilina 500mg",
                    "Loratadina 10mg", "Omeprazol 20mg", "Naproxeno 250mg",
                    "Metformina 850mg", "Losartán 50mg", "Clorfenamina 4mg",
                    "Diclofenaco Sódico 50mg", "Azitromicina 500mg", "Ácido Acetilsalicílico 100mg",
                    "Ranitidina 150mg", "Dextrometorfano Jarabe", "Cetirizina 10mg",
                    "Atorvastatina 20mg", "Enalapril 10mg", "Salbutamol Inhalador"],
        "precio_rango": (3.0, 60.0),
    },
    "Cuidado Personal": {
        "subcategorias": ["Higiene Corporal", "Higiene Bucal", "Higiene Femenina", "Desodorantes"],
        "nombres": ["Jabón Líquido Avena", "Jabón en Barra Glicerina", "Gel de Ducha Aloe Vera",
                    "Pasta Dental Triple Acción", "Enjuague Bucal Menta", "Cepillo Dental Suave",
                    "Toallas Higiénicas Nocturnas", "Protectores Diarios", "Tampones Regular",
                    "Desodorante Roll-On", "Desodorante en Barra", "Talco Corporal"],
        "precio_rango": (4.0, 40.0),
    },
    "Dermocosmética": {
        "subcategorias": ["Protección Solar", "Cuidado Facial", "Cuidado Capilar", "Antiedad"],
        "nombres": ["Protector Solar SPF50", "Crema Facial Hidratante", "Serum Vitamina C",
                    "Shampoo Anticaspa", "Acondicionador Reparador", "Crema Antiarrugas",
                    "Loción Corporal Hidratante", "Limpiador Facial Espumoso", "Contorno de Ojos",
                    "Mascarilla Facial Hidratante"],
        "precio_rango": (15.0, 130.0),
    },
    "Suplementos y Vitaminas": {
        "subcategorias": ["Multivitamínicos", "Vitamina C", "Omega 3", "Proteínas y Aminoácidos"],
        "nombres": ["Multivitamínico Adultos", "Vitamina C 1000mg", "Omega 3 1000mg",
                    "Complejo B", "Calcio + Vitamina D", "Colágeno Hidrolizado",
                    "Proteína Whey Vainilla", "Magnesio Quelado", "Zinc 50mg", "Hierro Quelado"],
        "precio_rango": (10.0, 95.0),
    },
    "Cuidado del Bebé": {
        "subcategorias": ["Pañales", "Toallitas Húmedas", "Fórmulas Infantiles", "Accesorios Bebé"],
        "nombres": ["Pañales Talla M x30", "Pañales Talla G x26", "Toallitas Húmedas x80",
                    "Fórmula Infantil Etapa 1", "Fórmula Infantil Etapa 2", "Shampoo para Bebé",
                    "Crema Antipañalitis", "Aceite Corporal para Bebé"],
        "precio_rango": (8.0, 85.0),
    },
    "Equipos Médicos": {
        "subcategorias": ["Termómetros", "Tensiómetros", "Nebulizadores", "Glucómetros"],
        "nombres": ["Termómetro Digital", "Tensiómetro Digital de Brazo", "Nebulizador Portátil",
                    "Glucómetro Digital", "Pulsioxímetro Digital", "Tiras Reactivas de Glucosa"],
        "precio_rango": (20.0, 350.0),
    },
    "Insumos Médicos": {
        "subcategorias": ["Mascarillas", "Guantes", "Alcohol y Antisépticos", "Gasas y Vendas"],
        "nombres": ["Mascarillas Quirúrgicas x50", "Guantes de Látex x100", "Alcohol en Gel 500ml",
                    "Alcohol Medicinal 70° 250ml", "Gasas Estériles", "Vendas Elásticas",
                    "Algodón 500g", "Esparadrapo Hipoalergénico"],
        "precio_rango": (2.0, 40.0),
    },
}

marcas_ficticias = ["MediFarma", "BioSana", "NovaSalud", "FarmaVida", "VitalCare",
                     "PharmaPlus", "SaludTotal Labs", "BienestarLab", "DermaPro",
                     "CardioSalud", "Genérico"]

categorias_list = list(catalogo_categorias.keys())
filas_producto = []
for i in range(1, N_PRODUCTOS + 1):
    categoria = categorias_list[(i - 1) % len(categorias_list)]
    info = catalogo_categorias[categoria]
    subcategoria = py_random.choice(info["subcategorias"])
    nombre_base = py_random.choice(info["nombres"])
    marca = py_random.choice(marcas_ficticias)
    lo, hi = info["precio_rango"]
    precio_lista = round(float(py_random.uniform(lo, hi)), 2)
    margen = py_random.uniform(0.30, 0.60)  # margen comercial
    costo = round(precio_lista * (1 - margen), 2)
    filas_producto.append((i, nombre_base, categoria, subcategoria, marca, precio_lista, costo))

dim_producto_clean = pd.DataFrame(
    filas_producto,
    columns=["id_producto", "nombre", "categoria", "subcategoria", "marca", "precio_lista", "costo"]
)

# --- Peso de popularidad tipo Pareto (esquema de 2 niveles: "estrella" vs "regular")
# Un Pareto puro (np.random.pareto) con cola muy pesada hace que UN solo producto
# concentre una parte absurda de las ventas. En vez de eso, se separan los productos
# en un grupo "estrella" (20%) con peso alto y un grupo "regular" (80%) con peso bajo,
# lo que reproduce el patrón ~20%/~80% sin que ningún producto individual domine
# de forma poco realista.
n_top_productos = int(N_PRODUCTOS * 0.20)
pop_producto = np.empty(N_PRODUCTOS)
pop_producto[:n_top_productos] = py_random.uniform(3.0, 7.0, size=n_top_productos)
pop_producto[n_top_productos:] = py_random.uniform(0.2, 0.8, size=N_PRODUCTOS - n_top_productos)
py_random.shuffle(pop_producto)  # para que no dependa del orden/id del producto
pop_producto = pop_producto / pop_producto.sum()
dim_producto_clean["_peso_popularidad"] = pop_producto  # se usa solo en memoria, no se exporta


# ==============================================================================
# 4. DIM_PROMOCION (~40 promociones)
# ==============================================================================
print("[4/6] Generando Dim_Promocion ...")

tipos_promo = ["Descuento Directo", "Descuento por Categoría", "2x1", "Cliente Frecuente", "Liquidación"]
nombres_promo_base = ["Campaña Salud", "Quincena Bienestar", "Semana del Ahorro", "Fiestas Patrias",
                      "Navidad Saludable", "Verano Saludable", "Día de la Madre", "Aniversario Botica",
                      "Black Friday Salud", "Regreso a Clases"]

filas_promo = []
for i in range(1, N_PROMOCIONES + 1):
    tipo = py_random.choice(tipos_promo)
    nombre = f"{py_random.choice(nombres_promo_base)} {py_random.randint(1, 99)}"
    descuento_pct = round(float(py_random.choice([0.05, 0.10, 0.15, 0.20, 0.25, 0.30])), 2)
    # duración de la promo: entre 5 y 30 días, distribuida en los 2 años de negocio
    dur = int(py_random.randint(5, 30))
    inicio_offset = int(py_random.randint(0, N_DIAS - dur - 1))
    f_inicio = FECHA_INICIO + timedelta(days=inicio_offset)
    f_fin = f_inicio + timedelta(days=dur)
    filas_promo.append((i, nombre, tipo, descuento_pct, f_inicio, f_fin))

dim_promocion_clean = pd.DataFrame(
    filas_promo,
    columns=["id_promocion", "nombre", "tipo", "descuento_pct", "fecha_inicio", "fecha_fin"]
)


# ==============================================================================
# 5. DIM_CLIENTE (~5000 clientes) + parámetros de churn/frecuencia (internos)
# ==============================================================================
print("[5/6] Generando Dim_Cliente ...")

sexos = py_random.choice(["F", "M"], size=N_CLIENTES, p=[0.58, 0.42])  # botica: más clientela femenina

nombres_clientes = []
for s in sexos:
    if s == "F":
        nombres_clientes.append(fake.name_female())
    else:
        nombres_clientes.append(fake.name_male())

distritos_lima = ["Miraflores", "San Isidro", "Surco", "La Molina", "San Borja",
                  "Lince", "Jesús María", "Pueblo Libre", "Magdalena del Mar",
                  "San Miguel", "Barranco", "Chorrillos", "Comas", "San Juan de Lurigancho",
                  "Los Olivos", "Independencia", "Ate", "Villa El Salvador",
                  "Santiago de Surco", "Breña", "Rímac", "San Martín de Porres"]

distrito_cliente = py_random.choice(distritos_lima, size=N_CLIENTES)

# Fecha de nacimiento: edades entre 18 y 90 años (referencia: fin del periodo)
edad_dias_min = 18 * 365
edad_dias_max = 90 * 365
fecha_nacimiento = [
    FECHA_FIN - timedelta(days=int(py_random.randint(edad_dias_min, edad_dias_max)))
    for _ in range(N_CLIENTES)
]

# Fecha de alta al programa: algunos clientes ya estaban antes del periodo de ventas analizado
fecha_alta = [
    FECHA_INICIO - timedelta(days=int(py_random.randint(-30, 365)))
    for _ in range(N_CLIENTES)
]
fecha_alta = [max(FECHA_INICIO - timedelta(days=365), f) for f in fecha_alta]

# Peso de frecuencia de compra: esquema de 3 niveles (muy frecuente / frecuente /
# esporádico) en vez de una Pareto pura, para evitar que un solo cliente concentre
# un número de visitas poco realista (p. ej. más de una compra por día durante 2 años).
# ~5% de clientes son "muy frecuentes", ~20% "frecuentes" y ~75% "esporádicos".
n_muy_frec = int(N_CLIENTES * 0.05)
n_frec = int(N_CLIENTES * 0.20)
n_esporadico = N_CLIENTES - n_muy_frec - n_frec

peso_frecuencia_cliente = np.empty(N_CLIENTES)
peso_frecuencia_cliente[:n_muy_frec] = py_random.uniform(8.0, 16.0, size=n_muy_frec)
peso_frecuencia_cliente[n_muy_frec:n_muy_frec + n_frec] = py_random.uniform(2.0, 6.0, size=n_frec)
peso_frecuencia_cliente[n_muy_frec + n_frec:] = py_random.uniform(0.05, 1.2, size=n_esporadico)
py_random.shuffle(peso_frecuencia_cliente)  # para que no dependa del orden/id del cliente
peso_frecuencia_cliente = peso_frecuencia_cliente / peso_frecuencia_cliente.sum()

# Segmento de programa correlacionado con la frecuencia (Pareto)
rank_frecuencia = pd.Series(peso_frecuencia_cliente).rank(pct=True).values
segmento_programa = np.select(
    [rank_frecuencia >= 0.95, rank_frecuencia >= 0.80, rank_frecuencia >= 0.50],
    ["Oro", "Plata", "Bronce"],
    default="Sin Programa"
)

# --- Flag interno de abandono (churn) simulado: ~20-25% de clientes ----------
es_churn_simulado = py_random.choice([True, False], size=N_CLIENTES, p=[0.225, 0.775])

ACTIVE_START = np.array([max(FECHA_INICIO, fa) for fa in fecha_alta])
ACTIVE_END = []
for i in range(N_CLIENTES):
    if es_churn_simulado[i]:
        inicio_i = ACTIVE_START[i]
        # margen para que exista historial de compra antes del corte
        min_corte = inicio_i + timedelta(days=30)
        max_corte = FECHA_FIN - timedelta(days=30)
        if min_corte >= max_corte:
            corte = max_corte
        else:
            dias_rango = (max_corte - min_corte).days
            corte = min_corte + timedelta(days=int(py_random.randint(0, max(dias_rango, 1))))
        ACTIVE_END.append(corte)
    else:
        ACTIVE_END.append(FECHA_FIN)
ACTIVE_END = np.array(ACTIVE_END)

dim_cliente_base = pd.DataFrame({
    "id_cliente": range(1, N_CLIENTES + 1),
    "nombre": nombres_clientes,
    "sexo": sexos,
    "fecha_nacimiento": fecha_nacimiento,
    "distrito": distrito_cliente,
    "fecha_alta": fecha_alta,
    "segmento_programa": segmento_programa,
})


# ==============================================================================
# 6. FACT_VENTAS (60,000 líneas) - estacionalidad, Pareto, afinidad de canasta,
#    promociones vigentes y churn
# ==============================================================================
print("[6/6] Generando Fact_Ventas (esto puede tardar unos segundos) ...")

# --- 6.1 Distribuir "canastas" (visitas de compra) entre clientes, según peso de frecuencia
AVG_BASKET_SIZE = 1.8
n_baskets_objetivo = int(round(N_VENTAS / AVG_BASKET_SIZE))

n_baskets_por_cliente = np.maximum(
    np.round(peso_frecuencia_cliente * n_baskets_objetivo).astype(int), 0
)
# Garantizar que existan "esporádicos" con 0 o 1 compra y algunos con muchas (ya lo da Pareto)

dias_array = dim_tiempo["fecha"].values  # numpy datetime64[ns] array, alineado con peso_estacional_dia

# --- 6.2 Tabla de afinidad de categorías (para lift > 1 en canasta) -----------
# Definimos pares de categorías que tienden a comprarse juntas en la misma visita:
pares_afinidad = [
    ("Medicamentos", "Suplementos y Vitaminas"),     # gripe -> vitamina C
    ("Cuidado del Bebé", "Cuidado Personal"),        # pañales -> higiene
    ("Dermocosmética", "Cuidado Personal"),          # cremas -> jabones
    ("Equipos Médicos", "Medicamentos"),             # tensiómetro -> medicamento cardiovascular
]
mapa_afinidad = {}
for a, b in pares_afinidad:
    mapa_afinidad.setdefault(a, []).append(b)
    mapa_afinidad.setdefault(b, []).append(a)

PROB_AFINIDAD = 0.45  # prob. de que el 2do/3er item de la canasta venga de la categoría afín

productos_por_categoria = dim_producto_clean.groupby("categoria")["id_producto"].apply(list).to_dict()
peso_por_categoria_dict = dim_producto_clean.groupby("categoria")["_peso_popularidad"].apply(list).to_dict()


def elegir_producto(categoria=None):
    """Elige un id_producto. Si se da categoria, restringe la elección a esa categoría
    (ponderado por popularidad); si no, elige sobre todo el catálogo (ponderado, Pareto)."""
    if categoria is not None and categoria in productos_por_categoria:
        ids = productos_por_categoria[categoria]
        pesos = np.array(peso_por_categoria_dict[categoria])
        pesos = pesos / pesos.sum()
        return py_random.choice(ids, p=pesos)
    else:
        return py_random.choice(dim_producto_clean["id_producto"].values,
                                 p=dim_producto_clean["_peso_popularidad"].values)


id_a_categoria = dim_producto_clean.set_index("id_producto")["categoria"].to_dict()
id_a_precio = dim_producto_clean.set_index("id_producto")["precio_lista"].to_dict()

promos_inicio = dim_promocion_clean["fecha_inicio"].values
promos_fin = dim_promocion_clean["fecha_fin"].values
promos_ids = dim_promocion_clean["id_promocion"].values
promos_pct = dim_promocion_clean["descuento_pct"].values


def promo_vigente(fecha_venta):
    """Devuelve (id_promocion, descuento_pct) de una promo vigente en esa fecha, o (None, 0.0)."""
    activa = (promos_inicio <= fecha_venta) & (fecha_venta <= promos_fin)
    idx_activas = np.where(activa)[0]
    if len(idx_activas) == 0:
        return None, 0.0
    # ~60% de probabilidad de que la promoción vigente realmente se aplique a esta línea
    if py_random.random() > 0.60:
        return None, 0.0
    elegido = py_random.choice(idx_activas)
    return int(promos_ids[elegido]), float(promos_pct[elegido])


lineas = []  # acumulador de líneas de venta (tuplas)

for cid in range(N_CLIENTES):
    n_baskets = n_baskets_por_cliente[cid]
    if n_baskets <= 0:
        continue

    ini, fin = ACTIVE_START[cid], ACTIVE_END[cid]
    mascara_ventana = (dias_array >= np.datetime64(ini)) & (dias_array <= np.datetime64(fin))
    idx_ventana = np.where(mascara_ventana)[0]
    if len(idx_ventana) == 0:
        continue
    pesos_ventana = peso_estacional_dia[idx_ventana]
    pesos_ventana = pesos_ventana / pesos_ventana.sum()

    fechas_canastas = py_random.choice(dias_array[idx_ventana], size=n_baskets, p=pesos_ventana)
    tiendas_canastas = py_random.choice(dim_tienda["id_tienda"].values, size=n_baskets)
    tamanos_canasta = py_random.choice([1, 2, 3, 4], size=n_baskets, p=[0.45, 0.30, 0.15, 0.10])

    id_cliente_real = cid + 1  # ids van de 1..N_CLIENTES

    for b in range(n_baskets):
        fecha_b = pd.Timestamp(fechas_canastas[b])
        tienda_b = int(tiendas_canastas[b])
        tam = int(tamanos_canasta[b])

        id_prod_primero = elegir_producto()
        cat_primero = id_a_categoria[id_prod_primero]
        productos_canasta = [id_prod_primero]

        for _ in range(tam - 1):
            usar_afinidad = (cat_primero in mapa_afinidad) and (py_random.random() < PROB_AFINIDAD)
            if usar_afinidad:
                cat_destino = py_random.choice(mapa_afinidad[cat_primero])
                pid = elegir_producto(categoria=cat_destino)
            else:
                pid = elegir_producto()
            productos_canasta.append(pid)

        id_promo_b, pct_b = promo_vigente(np.datetime64(fecha_b))

        for pid in productos_canasta:
            precio_lista_p = id_a_precio[pid]
            precio_unitario = round(float(precio_lista_p * py_random.uniform(0.95, 1.06)), 2)
            cantidad = int(py_random.choice([1, 2, 3], p=[0.7, 0.22, 0.08]))
            if id_promo_b is not None:
                descuento = round(precio_unitario * cantidad * pct_b, 2)
            else:
                descuento = 0.0
            lineas.append((fecha_b, id_cliente_real, pid, tienda_b, id_promo_b,
                            cantidad, precio_unitario, descuento))

# Recortar/ajustar a exactamente N_VENTAS líneas "base" (antes de duplicar)
if len(lineas) > N_VENTAS:
    idx_recorte = py_random.choice(len(lineas), size=N_VENTAS, replace=False)
    idx_recorte = np.sort(idx_recorte)
    lineas = [lineas[i] for i in idx_recorte]
elif len(lineas) < N_VENTAS:
    faltan = N_VENTAS - len(lineas)
    extra_idx = py_random.choice(len(lineas), size=faltan, replace=True)
    lineas = lineas + [lineas[i] for i in extra_idx]

fact_ventas_base = pd.DataFrame(
    lineas,
    columns=["fecha", "id_cliente", "id_producto", "id_tienda", "id_promocion",
             "cantidad", "precio_unitario", "descuento"]
)
fact_ventas_base = fact_ventas_base.sort_values("fecha").reset_index(drop=True)
fact_ventas_base.insert(0, "id_venta", range(1, len(fact_ventas_base) + 1))

print(f"   Líneas de venta base generadas: {len(fact_ventas_base):,}")


# ==============================================================================
# 7. ENRIQUECER DIM_CLIENTE (PROCESSED) CON ETIQUETA DE ABANDONO (CHURN)
# ==============================================================================
print("\nCalculando etiqueta de abandono (churn) a partir del historial real de compras ...")

ultima_compra = fact_ventas_base.groupby("id_cliente")["fecha"].max()
dim_cliente_clean = dim_cliente_base.copy()
dim_cliente_clean["fecha_ultima_compra"] = dim_cliente_clean["id_cliente"].map(ultima_compra)
dim_cliente_clean["dias_desde_ultima_compra"] = (
    FECHA_FIN - dim_cliente_clean["fecha_ultima_compra"]
).dt.days

# La etiqueta de abandono (target de clasificación) usa la lógica de generación real:
# un cliente es "churn" si su ventana de actividad se cerró antes del fin del periodo
# (es decir, se le dejó de asignar compras a propósito a partir de cierta fecha de corte),
# que es la verdadera causa de abandono simulada en el negocio. Las columnas de recencia
# (fecha_ultima_compra / dias_desde_ultima_compra) quedan disponibles como variables
# predictoras para un modelo de clasificación, en vez de usarse como la propia regla del label
# (un umbral fijo de "días sin comprar" no es confiable aquí porque muchos clientes son
# esporádicos por naturaleza, no porque hayan abandonado).
dim_cliente_clean["churn"] = es_churn_simulado.astype(int)

pct_churn_real = dim_cliente_clean["churn"].mean() * 100
print(f"   % de clientes etiquetados como abandono (churn=1): {pct_churn_real:.1f}%  "
      f"(objetivo de diseño: 20-25%)")


# ==============================================================================
# 8. INTRODUCIR PROBLEMAS DE CALIDAD (SOLO EN data/raw)
# ==============================================================================
print("\nIntroduciendo problemas de calidad en las versiones 'raw' ...")

# ------------------------------------------------------------------------------
# 8.1 Dim_Cliente RAW: nulos en distrito (~3-8%) + fechas en formato mixto
# ------------------------------------------------------------------------------
dim_cliente_raw = dim_cliente_base.copy()  # OJO: raw = columnas operativas originales (sin churn)

frac_nulos_distrito = py_random.uniform(0.03, 0.08)
idx_nulos_distrito = py_random.choice(N_CLIENTES, size=int(N_CLIENTES * frac_nulos_distrito), replace=False)
dim_cliente_raw.loc[idx_nulos_distrito, "distrito"] = np.nan

dim_cliente_raw["fecha_nacimiento"] = dim_cliente_raw["fecha_nacimiento"].apply(fecha_a_texto_mixto)
dim_cliente_raw["fecha_alta"] = dim_cliente_raw["fecha_alta"].apply(fecha_a_texto_mixto)

# ------------------------------------------------------------------------------
# 8.2 Dim_Producto RAW: nulos en categoria (~3-8%) + texto sucio + typos
# ------------------------------------------------------------------------------
dim_producto_raw = dim_producto_clean.drop(columns=["_peso_popularidad"]).copy()

frac_nulos_categoria = py_random.uniform(0.03, 0.08)
idx_nulos_categoria = py_random.choice(N_PRODUCTOS, size=int(N_PRODUCTOS * frac_nulos_categoria), replace=False)

# Texto sucio en categoria (mayúsculas/minúsculas/espacios) para el resto de filas
idx_restantes = np.setdiff1d(np.arange(N_PRODUCTOS), idx_nulos_categoria)
idx_sucio_categoria = py_random.choice(idx_restantes, size=int(len(idx_restantes) * 0.45), replace=False)


def ensuciar_texto_categoria(valor):
    estilo = py_random.choice(["mayus", "minus", "espacios", "mayus_espacios"])
    if estilo == "mayus":
        return valor.upper()
    elif estilo == "minus":
        return valor.lower()
    elif estilo == "espacios":
        return f"  {valor}  "
    else:
        return f" {valor.upper()} "


dim_producto_raw.loc[idx_sucio_categoria, "categoria"] = dim_producto_raw.loc[
    idx_sucio_categoria, "categoria"
].apply(ensuciar_texto_categoria)
dim_producto_raw.loc[idx_nulos_categoria, "categoria"] = np.nan

# --- Errores de tipeo en "nombre" (mayúsculas/minúsculas + tildes) -----------
idx_typo_nombre = py_random.choice(N_PRODUCTOS, size=int(N_PRODUCTOS * 0.18), replace=False)


def aplicar_typo_nombre(valor):
    estilo = py_random.choice(["mayus_total", "minus_total", "sin_tildes"])
    if estilo == "mayus_total":
        return valor.upper()
    elif estilo == "minus_total":
        return valor.lower()
    else:
        return strip_accents(valor)


dim_producto_raw.loc[idx_typo_nombre, "nombre"] = dim_producto_raw.loc[
    idx_typo_nombre, "nombre"
].apply(aplicar_typo_nombre)

# ------------------------------------------------------------------------------
# 8.3 Fact_Ventas RAW: outliers, claves huérfanas, fechas mixtas, nulos en
#     descuento, duplicados
# ------------------------------------------------------------------------------
fact_ventas_raw = fact_ventas_base.copy()
n_base = len(fact_ventas_raw)

# --- Outliers en cantidad y precio_unitario (~0.5% cada uno) -----------------
idx_outlier_cantidad = py_random.choice(n_base, size=int(n_base * 0.004), replace=False)
fact_ventas_raw.loc[idx_outlier_cantidad, "cantidad"] = py_random.choice(
    [150, 200, 500, 999], size=len(idx_outlier_cantidad)
)

idx_outlier_precio = py_random.choice(n_base, size=int(n_base * 0.004), replace=False)
fact_ventas_raw.loc[idx_outlier_precio, "precio_unitario"] = py_random.choice(
    [0.01, 0.50, 1999.0, 4999.0], size=len(idx_outlier_precio)
)

# --- Claves huérfanas: ~0.5% de id_producto que NO existen en Dim_Producto ----
idx_huerfanas = py_random.choice(n_base, size=int(n_base * 0.005), replace=False)
ids_falsos = py_random.choice(np.arange(N_PRODUCTOS + 1001, N_PRODUCTOS + 1100), size=len(idx_huerfanas))
fact_ventas_raw.loc[idx_huerfanas, "id_producto"] = ids_falsos
n_huerfanas_insertadas = len(idx_huerfanas)

# --- Nulos en descuento (~3-8%) ------------------------------------------------
frac_nulos_descuento = py_random.uniform(0.03, 0.08)
idx_nulos_descuento = py_random.choice(n_base, size=int(n_base * frac_nulos_descuento), replace=False)
fact_ventas_raw.loc[idx_nulos_descuento, "descuento"] = np.nan

# --- Fechas en formato mixto (texto) ------------------------------------------
fact_ventas_raw["fecha"] = fact_ventas_raw["fecha"].apply(fecha_a_texto_mixto)

# --- Duplicar ~1-2% de las líneas de venta ------------------------------------
frac_duplicados = py_random.uniform(0.01, 0.02)
idx_duplicar = py_random.choice(n_base, size=int(n_base * frac_duplicados), replace=False)
filas_duplicadas = fact_ventas_raw.loc[idx_duplicar].copy()
fact_ventas_raw = pd.concat([fact_ventas_raw, filas_duplicadas], ignore_index=True)

# Mezclar el orden final para que los duplicados no queden todos al final (más realista)
fact_ventas_raw = fact_ventas_raw.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_duplicados_reales = fact_ventas_raw.duplicated().sum()


# ==============================================================================
# 9. GUARDAR ARCHIVOS
# ==============================================================================
print("\nGuardando archivos CSV (separador ';', UTF-8) ...")

print("\n>> data/raw/  (versión con problemas de calidad)")
guardar_csv(dim_cliente_raw, f"{RAW_DIR}/Dim_Cliente.csv")
guardar_csv(dim_producto_raw, f"{RAW_DIR}/Dim_Producto.csv")
guardar_csv(dim_tienda, f"{RAW_DIR}/Dim_Tienda.csv")
guardar_csv(dim_tiempo, f"{RAW_DIR}/Dim_Tiempo.csv")
guardar_csv(dim_promocion_clean, f"{RAW_DIR}/Dim_Promocion.csv")
guardar_csv(fact_ventas_raw, f"{RAW_DIR}/Fact_Ventas.csv")

print("\n>> data/processed/  (versión limpia de las tablas Dim_*)")
guardar_csv(dim_cliente_clean, f"{PROCESSED_DIR}/Dim_Cliente.csv")
guardar_csv(dim_producto_clean.drop(columns=["_peso_popularidad"]), f"{PROCESSED_DIR}/Dim_Producto.csv")
guardar_csv(dim_tienda_clean, f"{PROCESSED_DIR}/Dim_Tienda.csv")
guardar_csv(dim_tiempo_clean, f"{PROCESSED_DIR}/Dim_Tiempo.csv")
guardar_csv(dim_promocion_clean, f"{PROCESSED_DIR}/Dim_Promocion.csv")


 GENERANDO DATOS SINTÉTICOS - BOTICA SALUD TOTAL
 Periodo de negocio: 2024-01-01 -> 2025-12-30  (730 días)

[1/6] Generando Dim_Tiempo ...
[2/6] Generando Dim_Tienda ...
[3/6] Generando Dim_Producto ...
[4/6] Generando Dim_Promocion ...
[5/6] Generando Dim_Cliente ...
[6/6] Generando Fact_Ventas (esto puede tardar unos segundos) ...
   Líneas de venta base generadas: 60,000

Calculando etiqueta de abandono (churn) a partir del historial real de compras ...
   % de clientes etiquetados como abandono (churn=1): 21.4%  (objetivo de diseño: 20-25%)

Introduciendo problemas de calidad en las versiones 'raw' ...

Guardando archivos CSV (separador ';', UTF-8) ...

>> data/raw/  (versión con problemas de calidad)
   -> Guardado: data/raw/Dim_Cliente.csv  (5,000 filas)
   -> Guardado: data/raw/Dim_Producto.csv  (500 filas)
   -> Guardado: data/raw/Dim_Tienda.csv  (15 filas)
   -> Guardado: data/raw/Dim_Tiempo.csv  (730 filas)
   -> Guardado: data/raw/Dim_Promocion.csv  (40 filas)
   -> Guardado

In [6]:
import pandas as pd
def resumen_calidad(nombre_tabla: str, df: pd.DataFrame, n_duplicados=None, n_huerfanas=None):
    """Imprime el resumen de calidad de una tabla: filas, % nulos, duplicados, huérfanas."""
    print(f"\n--- {nombre_tabla} ---")
    print(f"Filas: {len(df):,}   Columnas: {df.shape[1]}")
    pct_nulos = (df.isna().mean() * 100).round(2)
    pct_nulos = pct_nulos[pct_nulos > 0]
    if len(pct_nulos) > 0:
        print("% de nulos por columna (solo columnas con nulos):")
        for col, pct in pct_nulos.items():
            print(f"   - {col:<20s}: {pct:>6.2f} %")
    else:
        print("% de nulos por columna: 0% en todas las columnas")
    if n_duplicados is not None:
        print(f"Filas duplicadas (registro completo): {n_duplicados:,}")
    if n_huerfanas is not None:
        print(f"Claves huérfanas (id_producto inexistente en Dim_Producto): {n_huerfanas:,}")
# 10. RESUMEN DE CALIDAD INICIAL (sobre los archivos RAW)
# ==============================================================================
print("\n" + "=" * 80)
print(" RESUMEN DE CALIDAD INICIAL - ARCHIVOS data/raw/")
print("=" * 80)
 
resumen_calidad("Dim_Cliente(raw)", dim_cliente_raw)
resumen_calidad("Dim_Producto(raw)", dim_producto_raw)
resumen_calidad("Dim_Tienda(raw)", dim_tienda)
resumen_calidad("Dim_Tiempo(raw)", dim_tiempo)
resumen_calidad("Dim_Promocion(raw)", dim_promocion_clean)
resumen_calidad(
    "Fact_Ventas (raw)", fact_ventas_raw,
    n_duplicados=n_duplicados_reales,
    n_huerfanas=n_huerfanas_insertadas,
)
print("\nNota: los nulos en 'id_promocion' son lógica de negocio válida (no hubo promoción")
print("      vigente o no se aplicó), NO un problema de calidad introducido a propósito.")
print("      Los problemas de calidad deliberados están en: fechas mixtas, nulos en")
print("      'descuento'/'distrito'/'categoria', duplicados, texto sucio, typos,")
print("      outliers y claves huérfanas en 'id_producto'.")
 
print("\n" + "=" * 80)
print(" PROCESO COMPLETADO")
print("=" * 80)
print(f"Dim_Cliente   : {N_CLIENTES:,} filas")
print(f"Dim_Producto  : {N_PRODUCTOS:,} filas")
print(f"Dim_Tienda    : {N_TIENDAS:,} filas")
print(f"Dim_Tiempo    : {N_DIAS:,} filas")
print(f"Dim_Promocion : {N_PROMOCIONES:,} filas")
print(f"Fact_Ventas   : {len(fact_ventas_raw):,} filas (raw, incluye duplicados) "
      f"| {len(fact_ventas_base):,} filas base (antes de duplicar)")


 RESUMEN DE CALIDAD INICIAL - ARCHIVOS data/raw/

--- Dim_Cliente(raw) ---
Filas: 5,000   Columnas: 7
% de nulos por columna (solo columnas con nulos):
   - distrito            :   6.44 %

--- Dim_Producto(raw) ---
Filas: 500   Columnas: 7
% de nulos por columna (solo columnas con nulos):
   - categoria           :   6.20 %

--- Dim_Tienda(raw) ---
Filas: 15   Columnas: 5
% de nulos por columna: 0% en todas las columnas

--- Dim_Tiempo(raw) ---
Filas: 730   Columnas: 7
% de nulos por columna: 0% en todas las columnas

--- Dim_Promocion(raw) ---
Filas: 40   Columnas: 6
% de nulos por columna: 0% en todas las columnas

--- Fact_Ventas (raw) ---
Filas: 60,953   Columnas: 9
% de nulos por columna (solo columnas con nulos):
   - id_promocion        :  60.31 %
   - descuento           :   5.36 %
Filas duplicadas (registro completo): 953
Claves huérfanas (id_producto inexistente en Dim_Producto): 300

Nota: los nulos en 'id_promocion' son lógica de negocio válida (no hubo promoción
      vig